# 🍷 Wine Quality Prediction
**Oasis Infobyte SIP · Data Analytics · Level 2 · Task 2**

**Objective:** Train and compare three classifiers (Random Forest, SGD, SVC) to predict wine quality from physicochemical properties such as acidity, density and alcohol.

**Dataset:** Wine Quality dataset (UCI / Kaggle). Place the CSV next to this notebook, e.g. `winequality-red.csv` (UCI, `;` separated) or `WineQT.csv` (Kaggle, `,` separated). The loader below handles both.

## 1. Imports & setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid", palette="viridis")
RANDOM_STATE = 42

## 2. Load the dataset & inspect structure

In [ ]:
DATA_PATH = "winequality-red.csv"   # change to your file name (e.g. "WineQT.csv")

# sep=None lets pandas auto-detect ';' (UCI) or ',' (Kaggle)
df = pd.read_csv(DATA_PATH, sep=None, engine="python")
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
df = df.drop(columns=[c for c in ["id"] if c in df.columns])   # Kaggle version has an Id column

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

In [ ]:
# Class distribution of the original quality scores
quality_counts = df["quality"].value_counts().sort_index()
print(pd.DataFrame({"count": quality_counts,
                    "percent": (quality_counts / len(df) * 100).round(2)}))

plt.figure(figsize=(7, 4))
sns.countplot(x="quality", data=df)
plt.title("Distribution of wine quality scores")
plt.show()

**Observation:** The quality scores are concentrated in the middle of the scale (typically 5 and 6), while the extreme scores (3, 4, 8) have very few samples. The target is therefore strongly **imbalanced**.

## 3. Exploratory Data Analysis

In [ ]:
features = [c for c in df.columns if c != "quality"]

fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Observation:** Many features (e.g. `residual_sugar`, `chlorides`, `sulphates`, `total_sulfur_dioxide`) are right-skewed with long tails of outliers, while `density` and `pH` are roughly bell-shaped. Features are on very different scales, so scaling will matter for SGD and SVC.

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation heatmap")
plt.show()

print("Correlation with quality:")
print(df.corr()["quality"].drop("quality").sort_values(ascending=False).round(3))

**Observation:** `alcohol` shows the strongest positive correlation with quality, and `volatile_acidity` the strongest negative one. Several features are also correlated with each other (e.g. `fixed_acidity` with `citric_acid`, `density` and `pH`), which hints at some redundancy.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col in zip(axes, ["alcohol", "volatile_acidity", "sulphates"]):
    sns.boxplot(x="quality", y=col, data=df, ax=ax)
    ax.set_title(f"{col} by quality")
plt.tight_layout()
plt.show()

## 4. Class imbalance discussion
* Scores 5 and 6 make up the large majority of samples; scores 3, 4 and 8 are rare.
* A model trained on raw scores can reach decent accuracy simply by predicting 5 or 6 and will almost never predict rare grades. Rare classes also may not appear in the test set in sufficient numbers to evaluate reliably.
* **Mitigations used here:** (1) group scores into fewer, better-populated classes, (2) use a **stratified** train/test split, (3) use `class_weight="balanced"` so minority classes count more during training, and (4) report **macro/weighted F1** and per-class recall, not just accuracy.

## 5. Feature engineering: binning the target
Quality is grouped into **3 classes**:

| Class | Original scores | Rationale |
|---|---|---|
| Low | ≤ 5 | Below-average wines |
| Medium | 6 | The most common grade |
| High | ≥ 7 | Good/excellent wines, a small but business-relevant group |

*Justification:* individual scores 3, 4 and 8 are too rare to learn from. Merging them gives each class enough samples while keeping more information than a simple good/bad split.

In [ ]:
def bin_quality(q):
    if q <= 5:
        return "low"
    elif q == 6:
        return "medium"
    return "high"

df["quality_class"] = df["quality"].apply(bin_quality)
order = ["low", "medium", "high"]

counts = df["quality_class"].value_counts().reindex(order)
print(pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(2)}))

plt.figure(figsize=(6, 4))
sns.countplot(x="quality_class", data=df, order=order)
plt.title("Quality classes after binning")
plt.show()

## 6. Train/test split (stratified)

In [ ]:
X = df[features]
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nTrain class ratio:\n", y_train.value_counts(normalize=True).reindex(order).round(3))
print("\nTest class ratio:\n", y_test.value_counts(normalize=True).reindex(order).round(3))

The class ratios are almost identical in train and test, which confirms that stratification worked.

## 7. Model training
Three classifiers are trained. SGD and SVC are distance/gradient based, so they are wrapped in a pipeline with `StandardScaler`. Random Forest is scale-invariant and does not need scaling.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "SGD Classifier": make_pipeline(
        StandardScaler(),
        SGDClassifier(loss="hinge", class_weight="balanced", max_iter=2000,
                      tol=1e-3, random_state=RANDOM_STATE)),
    "SVC (RBF)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=RANDOM_STATE)),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    print(f"Trained: {name}")

## 8. Evaluation: accuracy, classification report, confusion matrix

In [ ]:
results = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    results.append({"Model": name, "Accuracy": acc,
                    "Macro F1": f1_macro, "Weighted F1": f1_weighted})

    print("=" * 60)
    print(name, f"| accuracy = {acc:.3f}")
    print(classification_report(y_test, y_pred, labels=order, zero_division=0))

    cm = confusion_matrix(y_test, y_pred, labels=order)
    ConfusionMatrixDisplay(cm, display_labels=order).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 9. Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color=sns.color_palette("viridis", len(importances)))
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

importances.sort_values(ascending=False).round(3)

**Observation:** `alcohol` typically ranks at or near the top, followed by `sulphates`, `volatile_acidity` and `total_sulfur_dioxide`. This is consistent with the correlation analysis above.

## 10. Model comparison

In [ ]:
comparison = pd.DataFrame(results).set_index("Model").round(3)
display(comparison.sort_values("Macro F1", ascending=False))

comparison.plot(kind="bar", figsize=(9, 5), rot=0)
plt.title("Model comparison")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

best = comparison["Macro F1"].idxmax()
print(f"Best model by macro F1: {best} "
      f"(accuracy={comparison.loc[best, 'Accuracy']}, macro F1={comparison.loc[best, 'Macro F1']})")

## 11. Conclusion
* **Best model:** see the comparison table above. On this dataset **Random Forest** normally performs best (read the winner from the table printed in section 10 and update this line if your result differs).
* **Why it suits deployment:** it captures non-linear relationships and feature interactions without needing scaling, is robust to the skewed features and outliers seen in the EDA, copes better with class imbalance, and provides feature importance so a winemaker can see which properties drive quality.
* **SGD (linear)** is fast and light but a linear boundary is too simple for this problem. **SVC (RBF)** is competitive but slower to train/predict on large data, needs scaling and is harder to interpret.
* **Limitations / next steps:** the medium class overlaps heavily with its neighbours so some confusion is expected; hyperparameter tuning (`GridSearchCV`), SMOTE oversampling, and combining the red and white wine datasets are natural improvements.